# Test IRIS Telescope Integration (Mixed Protocols)

This notebook is used for interactive testing of the `TreeIrisObservatory` implementation and the `Pilar` and `IrisCCD` connectors.

**What does this notebook do?**
1. Runs local mock servers in the background: TCP for Pilar, UDP for iriscom (`FakeIriscom` from `test/protocols/iris_ccd/fake_iriscom.py`, same packets and replies as the real iriscom).
2. Replaces the configuration to connect to `localhost` instead of real hardware.
3. Initializes the IRIS device tree.
4. Allows executing `get` and `put` commands directly from the cells.

In [ ]:
import asyncio
import logging
import sys
import os
from unittest.mock import MagicMock

# Add the path to the main project directory to see the obsrv modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../')))  # first: stdlib also has a 'test' package

from obsrv.tree_components.specialized_components.tree_iris import TreeIrisObservatory
from obsrv.ob_config import SingletonConfig

# Logging configuration (to see what is happening in the connectors)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("JupyterIris")

## 1. Hardware Mocks Definition
The following classes simulate the operation of physical hardware (Pilar Telescope and UDP Camera).

In [ ]:
class MockPilarServer:
    """A simple TCP server mocking the Pilar telescope."""
    def __init__(self, port=5950):
        self.server = None
        self.host = '127.0.0.1'
        self.port = port
        self.responses = {
            "GET OBJECT.EQUATORIAL.RA": ("OBJECT.EQUATORIAL.RA=10.5", True),
            "GET OBJECT.EQUATORIAL.DEC": ("OBJECT.EQUATORIAL.DEC=-20.0", True),
            "SET POINTING.SETUP.FOCUS.POSITION=2000": ("POINTING.SETUP.FOCUS.POSITION=2000", True),
        }

    async def handle_client(self, reader, writer):
        try:
            while True:
                data = await reader.readline()
                if not data: break
                line = data.decode().strip()
                if not line: continue
                
                parts = line.split(' ', 1)
                if len(parts) < 2: continue
                cmd_id, command = parts[0], parts[1]
                
                # Response logic
                if command in self.responses:
                    val, ok = self.responses[command]
                    writer.write(f"{cmd_id} {val}\n".encode())
                    status = "COMMAND COMPLETE" if ok else "COMMAND FAILED"
                    writer.write(f"{cmd_id} {status}\n".encode())
                else:
                    writer.write(f"{cmd_id} COMMAND COMPLETE\n".encode())
                await writer.drain()
        except Exception:
            pass

    async def start(self):
        self.server = await asyncio.start_server(self.handle_client, self.host, self.port)
        print(f"[MockPilar] Listening on TCP {self.host}:{self.port}")
        asyncio.create_task(self.server.serve_forever())


# iriscom stand-in: 1024-byte packets, replies "<time> <command>\nOKAY <message>" / "E<code> <message>",
# one command at a time (an exposure or a filter move blocks it, like the real iriscom).
from test.protocols.iris_ccd.fake_iriscom import FakeIriscom

## 2. Mock Startup and Configuration
We start the servers in the background and replace `SingletonConfig` so that IRIS points to `localhost`.

In [ ]:
import confuse
from unittest.mock import MagicMock
import asyncio
import logging

# --- BULLETPROOF MockCfg CLASS ---
class MockCfg:
    """
    Mocks the behavior of the confuse library.
    Does not throw errors on [] access, only on .get().
    """
    def __init__(self, value, exists=True, path="root"):
        self.value = value
        self._exists = exists
        self._path = path

    def __getitem__(self, key):
        # Confuse returns a View object even if the key does not exist
        new_path = f"{self._path}.{key}"
        if not self._exists:
            return MockCfg(None, exists=False, path=new_path)
        
        if isinstance(self.value, dict):
            if key not in self.value:
                # Key is missing, but we return an object (marked as non-existent)
                return MockCfg(None, exists=False, path=new_path)
            return MockCfg(self.value[key], path=new_path)
        
        # Attempt to go deeper into a value that is not a dictionary
        return MockCfg(None, exists=False, path=new_path)

    def get(self, template=None):
        if not self._exists:
            # Here we throw a specific confuse error that the code can catch
            raise confuse.exceptions.NotFoundError(f"Mock config not found: {self._path}")
        return self.value

    # Iteration methods (e.g., for component in config['components'])
    def keys(self):
        if self._exists and isinstance(self.value, dict):
            return self.value.keys()
        return []

    def __iter__(self):
        if self._exists and isinstance(self.value, dict):
            return iter(self.value)
        return iter([])

    def __contains__(self, item):
        if self._exists and isinstance(self.value, dict):
            return item in self.value
        return False
        
    def as_str_seq(self):
        # Emulate getting a list of strings (e.g., flags)
        val = self.get()
        if isinstance(val, list):
            return [str(v) for v in val]
        return []

# --- Start servers (protection against port in use error) ---
try:
    pilar_mock = MockPilarServer()
    await pilar_mock.start()
except OSError:
    print("Pilar mock is probably already running (port in use). Continuing.")
iris_mock = await FakeIriscom.start()
print(f"[FakeIriscom] Listening on UDP {iris_mock.address}")

# --- Full Test Configuration ---
# Like tree.iris in obsrv/config.yaml; no protocol at the observatory level - every component has its own.
test_config_raw = {
    'tree': {
        'iris-notebook': {
            'timeout_multiplier': 0.8
        },
        'iris': {
            'observatory': {
                'comment': 'Test IRIS Telescope',
                'flags': ['production'],
                'lon': -70.2,
                'lat': -24.6,
                'elev': 2800,
                'epoch': 2000,
                'style': {'color': '#FF0000'},
                'components': {
                    'mount': {
                        'kind': 'telescope', 'device_number': 0,
                        'protocol': 'pilar', 'address': '127.0.0.1:5950',
                        'type': 'az', 'slew_timeout': 5
                    },
                    'focuser': {
                        'kind': 'focuser', 'device_number': 0,
                        'protocol': 'pilar', 'address': '127.0.0.1:5950'
                    },
                    'camera': {
                        'kind': 'camera', 'device_number': 0,
                        'protocol': 'iris_ccd', 'address': iris_mock.address,
                        'sensorname': 'QUIRC', 'cameraxsize': 1024, 'cameraysize': 1024,
                        'exposuremin': 0.0, 'exposuremax': 3600.0,
                        'ccdtemperature': {'subject': 'tic.telemetry.iris.ccdtemperature',
                                           'value_path': ['measurements', 'temperature'],
                                           'timestamp_path': ['ts'], 'max_age': 60.0}
                    },
                    'filterwheel': {
                        'kind': 'filterwheel', 'device_number': 0,
                        'protocol': 'iris_ccd', 'address': iris_mock.address,
                        'filters': {'J': 8, 'H': 7, 'K': 5}
                    },
                    'dome': {
                        'kind': 'dome', 'device_number': 0,
                        'protocol': 'alpaca', 'address': 'http://127.0.0.1:11111/api/v1',
                        'slew_timeout': 10
                    }
                }
            }
        }
    }
}

# Wrap in MockCfg
mock_conf_obj = MockCfg(test_config_raw)

# Patch SingletonConfig
SingletonConfig.get_config = MagicMock(return_value=mock_conf_obj)
print("Configuration updated. MockCfg is ready for anything.")

## 3. IRIS Tree Initialization
We create the `TreeIrisObservatory` object. At this point, the connectors are being prepared (lazy loading).

In [ ]:
tree_iris = TreeIrisObservatory('iris-notebook', observatory_name='iris')
print("IRIS tree initialized. Available components:")
for child in tree_iris._observatory.children:
    print(f" - {child}")

## 4. Pilar Protocol Tests (Telescope)
We fetch `rightascension` and test query parallelism.

In [ ]:
# Simple fetch (we use children['mount'] instead of .mount)
# Depending on the Observatory implementation, access might be via children['mount']
mount = tree_iris._observatory.children['mount']
ra = await mount.get('rightascension')
print(f"Right Ascension: {ra} degrees (Mock returns 10.5h)")

In [ ]:
# Parallelism test
mount = tree_iris._observatory.children['mount']
tasks = [
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension')
]
results = await asyncio.gather(*tasks)
print(f"Parallel query results: {results}")

# PUT test (focuser movement)
focuser = tree_iris._observatory.children['focuser']
resp = await focuser.put('move', Position=2000)
print(f"Focuser response: {resp}")

In [ ]:
# PUT test (focuser movement)
resp = await tree_iris._observatory.focuser.put('move', Position=2000)
print(f"Focuser response: {resp}")

## 5. IrisCCD Protocol Tests (Camera)
The server forwards each command to iriscom and waits for its reply. `camerastate`, `imageready` and `lastimage` are kept by the server (iriscom has no status command). `ccdtemperature` comes from NATS and needs a running NATS server, so it is not tested here.

In [ ]:
camera = tree_iris._observatory.children['camera']
print(f"Connected (sync): {await camera.get('connected')}")
print(f"Binning: {await camera.get('binx')}x{await camera.get('biny')}, frame {await camera.get('numx')}x{await camera.get('numy')}")

# Exposure: et 1.0 + go; the reply carries the file names written by iriscom
exposure = asyncio.create_task(camera.put('startexposure', Duration=1.0, Light=True))
await asyncio.sleep(0.3)
print(f"During exposure: camerastate={await camera.get('camerastate')}, imageready={await camera.get('imageready')}")
image = await exposure
print(f"After exposure: camerastate={await camera.get('camerastate')}, imageready={await camera.get('imageready')}")
print(f"Files: {image['files']}")

## 6. IrisCCD Protocol Tests (Filter Wheel)
Filters are numbered 1-8 like in iriscom; only the slots listed in `filters` can be selected. The position is known to the server only after it moved the wheel (-1 before).

In [ ]:
filterwheel = tree_iris._observatory.children['filterwheel']
print(f"Names: {await filterwheel.get('names')}")
print(f"Position at start: {await filterwheel.get('position')}")
print(await filterwheel.put('position', Position='J'))
print(await filterwheel.put('position', Position=8))   # same filter: not moved (iriscom would turn a full circle)
print(f"Position: {await filterwheel.get('position')}")
print(f"Commands received by iriscom: {iris_mock.received}")

## 7. Cleanup
Stopping servers after tests.

In [ ]:
await tree_iris.stop()
pilar_mock.server.close()
await pilar_mock.server.wait_closed()
iris_mock.close()
print("Mocks stopped.")